# Phase 5 — Sequence Curation

Phase 4 pooled the homology-search candidates and orphan-taxa hits, classified
them with the domain filters, and wrote the confirmed set to
`true_orthologs.fasta`. This phase is **curation only** — it prepares a clean,
human-verifiable ortholog set. Redundancy reduction (CD-HIT) and the multiple
sequence alignment now live in later phases, so nothing here collapses or
reorders sequences; every input sequence is either kept, or removed by an
explicit, documented manual decision.

Three steps:

1. **Annotate & rename** — replace each bare accession header with
   `Genus|Accession|Species`. Taxonomy is curated through an editable table
   (`taxonomy_overrides.csv`): the notebook pre-fills a best guess from NCBI,
   you correct any wrong rows by hand to a consistent ICTV-style naming, and
   the notebook applies your table. Genus first makes the set easy to scan.
2. **Domain QC report** — score every sequence against PF02346 (A29L/A27) and
   PF06086 (A26/A30L) and write the bit scores to a table. This step **removes
   nothing**; it flags anomalies (below-threshold PF02346, or unexpected
   PF06086 hits) so they can be reviewed by eye.
3. **Manual exclusion & rename** — apply the documented `EXCLUSIONS` and
   `RENAMES` from `manual_exclusions.py` to produce the final curated set. All
   removals and relabels happen here, in one auditable place, informed by the QC
   report. (Malformed `sp|X|Y` headers are auto-cleaned earlier, in Step 1.)

Because the manual list is informed by the QC report, the natural workflow is
to run steps 1–2, read `qc_scores.csv`, add any bad accessions (with reasons)
to `manual_exclusions.py`, then rerun so step 3 produces the final file.

**Steps**

0. Setup & configuration
1. Annotate & rename sequences
2. Domain QC report (flag only, remove nothing)
3. Apply manual exclusions & renames

## Step 0 — Setup & configuration

Taxonomy lookups reuse the shared `ncbi_fetch_utils` module (same courtesy
parameters and retry logic as Phases 1 and 3); FASTA and domain-table parsing
reuse `seqio_utils`; the manual-exclusion list is imported from
`manual_exclusions.py`. The two Pfam profiles (`PF02346.hmm`, `PF06086.hmm`)
must be downloaded from InterPro/Pfam and placed beside the notebook. External
tools run through the shared `run_command` helper.

In [1]:
from __future__ import annotations

import csv
import json
import logging
import os
import subprocess
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import ncbi_fetch_utils as ncbi
from ncbi_fetch_utils import FetchError
from seqio_utils import (
    parse_fasta_records,
    load_accession_organisms,
    parse_domtbl_scores,
    clean_accession,
)
from manual_exclusions import EXCLUSIONS, RENAMES

In [2]:
@dataclass(frozen=True)
class Config:
    phase4_true_orthologs_fasta: Path = Path("output_phase4/true_orthologs.fasta")
    accession_map_csv: Path = Path("output_phase1/accession_map.csv")
    pf02346_hmm: Path = Path("PF02346.hmm")
    pf06086_hmm: Path = Path("PF06086.hmm")

    wsl_prefix: list[str] = field(default_factory=lambda: ["wsl"])

    out_dir: Path = field(default_factory=lambda: Path("output_phase5"))

    @property
    def taxonomy_overrides_csv(self) -> Path:
        return self.out_dir / "taxonomy_overrides.csv"

    @property
    def annotated_fasta(self) -> Path:
        return self.out_dir / "annotated_orthologs.fasta"

    @property
    def qc_scores_csv(self) -> Path:
        return self.out_dir / "qc_scores.csv"

    @property
    def curated_fasta(self) -> Path:
        return self.out_dir / "manually_curated_orthologs.fasta"


CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)

ncbi.configure(
    email=os.environ.get("NCBI_EMAIL", ""),
    tool="poxvirus_pipeline",
    api_key=os.environ.get("NCBI_API_KEY", ""),
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(CFG.out_dir / "phase5.log"),
    ],
)
log = logging.getLogger("phase5")

if not CFG.phase4_true_orthologs_fasta.exists():
    raise FileNotFoundError(
        f"{CFG.phase4_true_orthologs_fasta} not found — run Phase 4 first."
    )

_missing_hmm = [p for p in (CFG.pf02346_hmm, CFG.pf06086_hmm) if not p.exists()]
if _missing_hmm:
    log.warning(
        "QC profiles not found (%s); Step 2 will fail until PF02346.hmm and "
        "PF06086.hmm are placed beside the notebook.", [str(p) for p in _missing_hmm]
    )

In [3]:
class CommandError(RuntimeError):
    """Raised when an external tool exits with a non-zero status."""


def run_command(cmd: list[str], log_file: Path) -> None:
    log.info("Running: %s", " ".join(cmd))
    with open(log_file, "w", encoding="utf-8") as fh:
        result = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        raise CommandError(
            f"Command failed (exit {result.returncode}): {' '.join(cmd)}. "
            f"See {log_file} for details."
        )

## Step 1 — Annotate & rename sequences (curated via editable table)

Taxonomy is curated through a two-pass, human-in-the-loop table so the final
names are hand-verified and consistently styled, rather than inheriting NCBI's
mix of common names and ICTV binomials.

**Pass 1 (first run).** For every accession the notebook derives a best guess:
it recovers the organism (from Phase 1's `accession_map.csv`, then NCBI, then
UniProt by accession) and resolves its genus from the NCBI lineage. It writes
`taxonomy_overrides.csv` with the raw NCBI values in read-only reference columns
(`ncbi_genus`, `ncbi_species`) and the same values pre-filled into the editable
`genus` and `species` columns.

**You edit.** Open `taxonomy_overrides.csv`, correct any wrong `genus`/`species`
cells to your chosen standard (ICTV binomial style, e.g. genus `Orthopoxvirus`,
species `Orthopoxvirus_camelpox`), and save. The `ncbi_*` columns stay as a
record of what NCBI reported; the `notes` column is yours.

**Pass 2 (rerun).** The notebook reads the table back and writes the annotated
FASTA using **your** `genus`/`species` values. Once `taxonomy_overrides.csv`
exists it is never regenerated, so a rerun can't overwrite your edits — delete
the file if you want to rebuild the pre-fill from scratch.

In [4]:
def resolve_organism(acc: str, acc_to_orgs: dict) -> tuple[str, str]:
    org = next(iter(acc_to_orgs.get(acc, set())), "")
    if org:
        return org, "map"
    try:
        org = ncbi.fetch_organism_by_accession(acc)
    except FetchError as e:
        log.error("NCBI organism fetch failed for %s: %s", acc, e)
        org = None
    if org:
        return org, "ncbi_by_accession"
    try:
        org = ncbi.fetch_organism_from_uniprot(acc)
    except FetchError as e:
        log.error("UniProt organism fetch failed for %s: %s", acc, e)
        org = None
    if org:
        return org, "uniprot_by_accession"
    return "", "unresolved"


records = parse_fasta_records(CFG.phase4_true_orthologs_fasta)
records = [(clean_accession(hdr), seq) for hdr, seq in records]
accessions = [hdr for hdr, _ in records]
_cleaned = [(a, h) for (h, _), a in zip(parse_fasta_records(CFG.phase4_true_orthologs_fasta), accessions) if a != h.split()[0]]
if _cleaned:
    log.info("Cleaned %d malformed header(s) to bare accession, e.g. %s -> %s",
             len(_cleaned), _cleaned[0][1].split()[0], _cleaned[0][0])

if CFG.taxonomy_overrides_csv.exists():
    log.info("%s exists — using your curated taxonomy (not regenerating).", CFG.taxonomy_overrides_csv)
    overrides = {}
    with open(CFG.taxonomy_overrides_csv, encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            overrides[row["accession"]] = {
                "genus": (row.get("genus") or "").strip() or "unclassified",
                "species": (row.get("species") or "").strip() or "unclassified",
            }
    missing_rows = [a for a in accessions if a not in overrides]
    if missing_rows:
        log.warning("%d accession(s) missing from the override table (labelled unclassified): %s",
                    len(missing_rows), missing_rows)
else:
    log.info("Building taxonomy pre-fill (first pass) ...")
    acc_to_orgs = load_accession_organisms(CFG.accession_map_csv)

    acc_org = {}
    for acc in accessions:
        org, source = resolve_organism(acc, acc_to_orgs)
        acc_org[acc] = (org, source)
        if source not in ("map",):
            log.info("Recovered organism for %s via %s: %s", acc, source, org or "<none>")

    unique_orgs = sorted({org for org, _ in acc_org.values() if org})
    org_to_taxid = {}
    for org in unique_orgs:
        try:
            tid = ncbi.esearch_taxid_for_organism(org)
        except FetchError as e:
            log.error("Taxid lookup failed for %s: %s", org, e)
            tid = None
        if tid:
            org_to_taxid[org] = tid
    lineage = ncbi.fetch_taxonomy_lineage(sorted(set(org_to_taxid.values())))

    overrides = {}
    with open(CFG.taxonomy_overrides_csv, "w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(["accession", "ncbi_genus", "ncbi_species", "genus", "species", "provenance", "notes"])
        for acc in accessions:
            org, source = acc_org[acc]
            tid = org_to_taxid.get(org, "")
            rec = lineage.get(tid, {})
            genus = rec.get("genus") or "unclassified"
            species = (org or "unclassified").replace(" ", "_")
            if genus == "unclassified" and org:
                source = "no_genus_in_lineage"
            writer.writerow([acc, genus, species, genus, species, source, ""])
            overrides[acc] = {"genus": genus, "species": species}

    log.warning(
        "Pre-fill written to %s — REVIEW and correct the 'genus'/'species' columns, "
        "then RERUN this cell to apply your edits.", CFG.taxonomy_overrides_csv,
    )

with open(CFG.annotated_fasta, "w", encoding="utf-8") as out:
    for acc, seq in records:
        a = overrides.get(acc, {"genus": "unclassified", "species": "unclassified"})
        out.write(f">{a['genus']}|{acc}|{a['species']}\n{seq}\n")

acc_annotation = overrides
genus_counts = Counter(a["genus"] for a in overrides.values())
log.info("Annotated %d sequences across %d genera: %s",
         len(overrides), len(genus_counts), dict(genus_counts))
_unclassified = sorted(a for a, v in overrides.items() if v["genus"] == "unclassified")
if _unclassified:
    log.warning("%d sequence(s) have genus=unclassified (edit the table if wrong): %s",
                len(_unclassified), _unclassified)

2026-07-23 00:29:15,722 [INFO] Cleaned 4 malformed header(s) to bare accession, e.g. sp|P16717.1|VFUS_SHEVK -> P16717.1
2026-07-23 00:29:15,728 [INFO] output_phase5\taxonomy_overrides.csv exists — using your curated taxonomy (not regenerating).
2026-07-23 00:29:15,737 [INFO] Annotated 83 sequences across 16 genera: {'Capripoxvirus': 10, 'Leporipoxvirus': 4, 'Orthopoxvirus': 26, 'Yatapoxvirus': 3, 'Sciuripoxvirus': 1, 'Cervidpoxvirus': 4, 'Oryzopoxvirus': 2, 'Centapoxvirus': 3, 'Parapoxvirus': 18, 'Pteropopoxvirus': 1, 'Vespertilionpoxvirus': 2, 'Mustelpoxvirus': 1, 'Suipoxvirus': 1, 'unclassified': 4, 'Crocodylidpoxvirus': 2, 'Avipoxvirus': 1}
2026-07-23 00:29:15,740 [WARNING] 4 sequence(s) have genus=unclassified (edit the table if wrong): ['QDJ95055.1', 'QHG62676.1', 'QHR82657.1', 'WGO62741.1']


## Step 2 — Domain QC report (flag only, remove nothing)

Scores every annotated sequence against both profiles and writes a per-sequence
table. Each profile is scanned twice: once with `--cut_ga` (Pfam's curated
gathering threshold) for the authoritative pass/fail, and once without a cutoff
so a sequence failing GA still shows its actual score rather than a blank. A
true A29L ortholog is expected to pass PF02346 and fail PF06086; sequences that
deviate are logged as warnings. **This step removes nothing** — it produces
`qc_scores.csv` for review, which informs what you add to `manual_exclusions.py`.

In [5]:
annotated = parse_fasta_records(CFG.annotated_fasta)
seq_by_acc = {hdr.split("|")[1]: seq for hdr, seq in annotated}
header_by_acc = {hdr.split("|")[1]: hdr for hdr, seq in annotated}


def hmmsearch_scores(hmm_path: Path, tag: str, use_cut_ga: bool) -> dict[str, float]:
    suffix = "ga" if use_cut_ga else "raw"
    domtbl = CFG.out_dir / f"qc_{tag}_{suffix}.domtbl"
    if not domtbl.exists():
        cmd = CFG.wsl_prefix + ["hmmsearch", "--domtblout", domtbl.as_posix(),
                                "-o", (CFG.out_dir / f"qc_{tag}_{suffix}.log").as_posix()]
        if use_cut_ga:
            cmd.append("--cut_ga")
        cmd += [hmm_path.as_posix(), CFG.annotated_fasta.as_posix()]
        run_command(cmd, log_file=CFG.out_dir / f"qc_{tag}_{suffix}_run.log")
    return parse_domtbl_scores(domtbl)


a29_raw = hmmsearch_scores(CFG.pf02346_hmm, "PF02346", use_cut_ga=False)
a29_ga = hmmsearch_scores(CFG.pf02346_hmm, "PF02346", use_cut_ga=True)
a26_raw = hmmsearch_scores(CFG.pf06086_hmm, "PF06086", use_cut_ga=False)
a26_ga = hmmsearch_scores(CFG.pf06086_hmm, "PF06086", use_cut_ga=True)


def score_key(acc: str, scores: dict[str, float]) -> float | None:
    if acc in scores:
        return scores[acc]
    return scores.get(header_by_acc.get(acc, acc))


with open(CFG.qc_scores_csv, "w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow([
        "genus", "accession", "species", "length",
        "PF02346_bits", "PF02346_pass_GA", "PF06086_bits", "PF06086_pass_GA", "flag",
    ])
    for hdr, seq in sorted(annotated, key=lambda x: x[0]):
        genus, acc, species = (hdr.split("|") + ["", "", ""])[:3]
        a29 = score_key(acc, a29_raw)
        a26 = score_key(acc, a26_raw)
        pass29 = acc in a29_ga or header_by_acc.get(acc) in a29_ga
        pass26 = acc in a26_ga or header_by_acc.get(acc) in a26_ga
        flag = ""
        if not pass29:
            flag = "PF02346_below_GA"
        elif pass26:
            flag = "PF06086_hit"
        writer.writerow([
            genus, acc, species, len(seq),
            f"{a29:.1f}" if a29 is not None else "NA", "Y" if pass29 else "N",
            f"{a26:.1f}" if a26 is not None else "NA", "Y" if pass26 else "N",
            flag,
        ])

accs = [hdr.split("|")[1] for hdr, _ in annotated]
n_a29_pass = sum(1 for a in accs if a in a29_ga or header_by_acc.get(a) in a29_ga)
n_a26_pass = sum(1 for a in accs if a in a26_ga or header_by_acc.get(a) in a26_ga)
a29_fail = sorted(a for a in accs if not (a in a29_ga or header_by_acc.get(a) in a29_ga))
a26_hit = sorted(a for a in accs if a in a26_ga or header_by_acc.get(a) in a26_ga)

log.info("Domain QC (%d sequences, nothing removed):", len(accs))
log.info("  PF02346 (A29L) pass GA: %d/%d", n_a29_pass, len(accs))
log.info("  PF06086 (A26)  pass GA: %d/%d", n_a26_pass, len(accs))
if a29_fail:
    log.warning("  below PF02346 GA (review): %s", a29_fail)
if a26_hit:
    log.warning("  unexpected PF06086 hit (review): %s", a26_hit)
log.info("QC report written to %s — review it, then edit manual_exclusions.py.", CFG.qc_scores_csv)

2026-07-23 00:29:33,146 [INFO] Running: wsl hmmsearch --domtblout output_phase5/qc_PF02346_raw.domtbl -o output_phase5/qc_PF02346_raw.log PF02346.hmm output_phase5/annotated_orthologs.fasta
2026-07-23 00:29:33,747 [INFO] Running: wsl hmmsearch --domtblout output_phase5/qc_PF02346_ga.domtbl -o output_phase5/qc_PF02346_ga.log --cut_ga PF02346.hmm output_phase5/annotated_orthologs.fasta
2026-07-23 00:29:34,110 [INFO] Running: wsl hmmsearch --domtblout output_phase5/qc_PF06086_raw.domtbl -o output_phase5/qc_PF06086_raw.log PF06086.hmm output_phase5/annotated_orthologs.fasta
2026-07-23 00:29:34,391 [INFO] Running: wsl hmmsearch --domtblout output_phase5/qc_PF06086_ga.domtbl -o output_phase5/qc_PF06086_ga.log --cut_ga PF06086.hmm output_phase5/annotated_orthologs.fasta
2026-07-23 00:29:34,665 [INFO] Domain QC (83 sequences, nothing removed):
2026-07-23 00:29:34,667 [INFO]   PF02346 (A29L) pass GA: 81/83
2026-07-23 00:29:34,669 [INFO]   PF06086 (A26)  pass GA: 0/83
2026-07-23 00:29:34,670 [WA

## Step 3 — Apply manual exclusions and renames

Applies the documented curation from `manual_exclusions.py`, in order:
**exclusions first**, then **renames** on what remains. Matching is by accession
(parsed from the `Genus|Accession|Species` header), so the manual file stays
simple — list plain accessions.

Renames relabel one accession as another. Because a rename changes the stated
provenance of a sequence — the header will carry the new accession while the
sequence itself is unchanged — each rename is logged as a warning, and should
only be used once the two records have been verified to hold an identical
sequence. Malformed UniProt-style headers (`sp|X|Y`) are cleaned to their bare
accession automatically in Step 1 and need no rename entry. This is the only
step that removes or relabels sequences; the final set is written to
`manually_curated_orthologs.fasta`.

In [6]:
annotated = parse_fasta_records(CFG.annotated_fasta)

kept = []
excluded_here = set()
for hdr, seq in annotated:
    acc = hdr.split("|")[1]
    if acc in EXCLUSIONS:
        log.info("Excluded %s — %s", acc, EXCLUSIONS[acc])
        excluded_here.add(acc)
    else:
        kept.append((hdr, seq))

for acc, reason in EXCLUSIONS.items():
    if acc not in excluded_here:
        log.info("Exclusion %s not present in set — nothing to remove.", acc)

renamed = []
final = []
for hdr, seq in kept:
    genus, acc, species = (hdr.split("|") + ["", "", ""])[:3]
    if acc in RENAMES:
        new_acc = RENAMES[acc]
        log.warning("Renamed %s -> %s (provenance change — verify sequences are identical)", acc, new_acc)
        renamed.append((acc, new_acc))
        acc = new_acc
    final.append((f"{genus}|{acc}|{species}", seq))

for old_acc in RENAMES:
    if old_acc not in {o for o, _ in renamed} and old_acc not in excluded_here:
        log.info("Rename %s not present in set — nothing to rename.", old_acc)

with open(CFG.curated_fasta, "w", encoding="utf-8") as out:
    for hdr, seq in final:
        out.write(f">{hdr}\n{seq}\n")

log.info("Final curated set: %d sequences (%d excluded, %d renamed) -> %s",
         len(final), len(excluded_here), len(renamed), CFG.curated_fasta)
log.info("Phase 5 complete. Outputs in %s", CFG.out_dir)

2026-07-23 00:29:50,673 [INFO] Excluded Q83970 — <Excluded due to wrong length, Cowpox virus>
2026-07-23 00:29:50,675 [INFO] Excluded Q8QMS8 — <Excluded due to wrong length, Cowpox virus>
2026-07-23 00:29:50,677 [INFO] Excluded QGT49194.1 — <Crocodylidpoxvirus below threshold GA in PF02346 QC check>
2026-07-23 00:29:50,679 [INFO] Excluded QGT49408.1 — <Crocodylidpoxvirus below threshold GA in PF02346 QC check>
2026-07-23 00:29:50,680 [INFO] Excluded XSG12557.1 — <Excluded due to frame-shifted C-term>
2026-07-23 00:29:50,681 [INFO] Excluded YCL31741.1 — <Excluded due to wrong length, Cowpox virus>
2026-07-23 00:29:50,683 [INFO] Excluded YFT67137.1 — <Excluded due to missing 3 first residues, Monkeypox virus>
2026-07-23 00:29:50,685 [INFO] Exclusion QJX15523.1 not present in set — nothing to remove.
2026-07-23 00:29:50,686 [INFO] Exclusion VTU03264.1 not present in set — nothing to remove.
2026-07-23 00:29:50,690 [INFO] Final curated set: 76 sequences (7 excluded, 0 renamed) -> output_ph